In [13]:
import pandas as pd
import spacy
from collections import Counter, defaultdict
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
import itertools

df = pd.read_csv("fakeTelegram.BR_2022.short.csv")
df = df.dropna(subset=["text_content_anonymous"])

nlp = spacy.load("pt_core_news_sm")
docs = list(nlp.pipe(df["text_content_anonymous"].astype(str), batch_size=32))

df["entidades"] = [
    list(set(ent.text.strip() for ent in doc.ents if len(ent.text.strip()) > 2))
    for doc in docs
]

all_ents = [ent for lista in df["entidades"] for ent in lista]
ent_count = Counter(all_ents)
ent_validas = set([ent for ent, count in ent_count.items() if count >= 5])

df["entidades_filtradas"] = df["entidades"].apply(lambda lista: [e for e in lista if e in ent_validas])
df = df[df["entidades_filtradas"].apply(lambda x: len(x) >= 2)]

te = TransactionEncoder()
te_ary = te.fit(df["entidades_filtradas"]).transform(df["entidades_filtradas"])
df_trans = pd.DataFrame(te_ary, columns=te.columns_)

fp_freq = fpgrowth(df_trans, min_support=0.005, use_colnames=True)
regras_fp = pd.DataFrame()
if not fp_freq.empty:
    regras_fp = association_rules(fp_freq, metric="lift", min_threshold=1.0)

df_trans["ent_count"] = df["entidades_filtradas"].apply(len)
df_sample = df_trans.sort_values("ent_count", ascending=False).drop("ent_count", axis=1).head(20)

apriori_freq = apriori(df_sample, min_support=0.01, use_colnames=True)
regras_ap = pd.DataFrame()
if not apriori_freq.empty:
    regras_ap = association_rules(apriori_freq, metric="lift", min_threshold=1.0)

def eclat(transactions, min_support=0.01):
    item_tidset = defaultdict(set)
    for tid, trans in enumerate(transactions):
        for item in trans:
            item_tidset[item].add(tid)

    frequent_items = {item: tids for item, tids in item_tidset.items()
                      if len(tids) / len(transactions) >= min_support}

    patterns = {}
    for size in range(2, 4):
        for combo in itertools.combinations(frequent_items.keys(), size):
            tids = set.intersection(*(frequent_items[item] for item in combo))
            support = len(tids) / len(transactions)
            if support >= min_support:
                patterns[combo] = support
    return patterns

trans_list = df["entidades_filtradas"].tolist()
eclat_result = eclat(trans_list, min_support=0.01)
eclat_df = pd.DataFrame(
    [(list(k), v) for k, v in eclat_result.items()],
    columns=["itemset", "support"]
)

print("\n📊 FP-Growth - Top 5 Regras:")
if not regras_fp.empty:
    print(regras_fp[["antecedents", "consequents", "support", "confidence", "lift"]].head())
else:
    print("⚠️ Nenhuma regra encontrada.")

print("\n📊 Apriori (amostra) - Top 5 Regras:")
if not regras_ap.empty:
    print(regras_ap[["antecedents", "consequents", "support", "confidence", "lift"]].head())
else:
    print("⚠️ Nenhuma regra encontrada.")

print("\n📊 ECLAT - Top 5 Padrões:")
if not eclat_df.empty:
    print(eclat_df.sort_values("support", ascending=False).head())
else:
    print("⚠️ Nenhum padrão frequente encontrado.")


📊 FP-Growth - Top 5 Regras:
  antecedents consequents   support  confidence      lift
0     (VÍDEO)      (VEJA)  0.846154    1.000000  1.181818
1      (VEJA)     (VÍDEO)  0.846154    1.000000  1.181818
2    (Brasil)      (VEJA)  0.153846    1.000000  1.181818
3      (VEJA)    (Brasil)  0.153846    0.181818  1.181818
4     (VÍDEO)    (Brasil)  0.153846    0.181818  1.181818

📊 Apriori (amostra) - Top 5 Regras:
                      antecedents                     consequents   support  \
0  (https://youtu.be/qbTzhB0akt8)                     (Bolsonaro)  0.153846   
1                     (Bolsonaro)  (https://youtu.be/qbTzhB0akt8)  0.153846   
2                        (Brasil)                          (VEJA)  0.153846   
3                          (VEJA)                        (Brasil)  0.153846   
4                         (VÍDEO)                        (Brasil)  0.153846   

   confidence      lift  
0    1.000000  6.500000  
1    1.000000  6.500000  
2    1.000000  1.181818  
3    0.